# Tests complémentaires (à lancer sur Colab, GPU)

Ce notebook fait seulement les runs. Les analyses sont dans `mlp_fmnist.ipynb` et `cnn_fmnist.ipynb` (elles relisent les CSV), sauf le pont p = 0.3 qui est analysé ici.

Ordre de priorité :
1. MLP largeur 16, 50 epochs : grille relancée avec le `train.py` actuel (+ frac 0.05)
2. CNN largeur 4, 50 epochs : frac 0.05 (ou toute la grille si les anciens runs n'ont pas `train_acc`)
3. Pont p = 0.3 (comparaison avec CIFAR-10) : MLP et CNN, frac 0.1 / 0.3 / 1
4. Calibration MLP largeur 32, 50 epochs
5. Contrôle CNN largeur 16, 50 epochs
6. (option) MLP avec un seul dropout avant la sortie, à p = 0.3

Chaque run écrit son CSV sur le Drive dès qu'il est fini et les runs déjà faits sont sautés : si Colab coupe, on relance les cellules et ça reprend où ça s'était arrêté. On pousse sur GitHub à la fin (ou quand on veut avec la dernière cellule).

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
%cd /content/drive/MyDrive/early-dropout
# si le pull râle à cause de modifs locales sur un notebook : !git checkout -- cnn_fmnist.ipynb  puis relancer
!git pull

In [ ]:
import os, glob, importlib
import pandas as pd
import torch
from torch import nn
from torchvision import datasets

import config, models, train
importlib.reload(config); importlib.reload(models); importlib.reload(train)
from config import BATCH, EARLY_GRID
from models import mlp, cnn
from train import entrainer

assert 0.05 in EARLY_GRID, "config.py pas à jour : faire le git pull"
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device, "| grille :", EARLY_GRID)

In [ ]:
tr = datasets.FashionMNIST("./data", train=True, download=True)
te = datasets.FashionMNIST("./data", train=False, download=True)
Xtr, Xte = tr.data.float().unsqueeze(1) / 255, te.data.float().unsqueeze(1) / 255
ytr, yte = tr.targets, te.targets

# même split et même normalisation que dans les notebooks
perm = torch.randperm(len(Xtr), generator=torch.Generator().manual_seed(0))
Xva, yva = Xtr[perm[:10000]], ytr[perm[:10000]]
Xtr, ytr = Xtr[perm[10000:]], ytr[perm[10000:]]
mu, sd = Xtr.mean(), Xtr.std()
Xtr, Xva, Xte = (Xtr - mu) / sd, (Xva - mu) / sd, (Xte - mu) / sd

img = {k: v.to(device) for k, v in dict(Xtr=Xtr, ytr=ytr, Xva=Xva, yva=yva, Xte=Xte, yte=yte).items()}  # CNN
plat = {k: (v.flatten(1) if k.startswith("X") else v) for k, v in img.items()}                          # MLP

## Un run = un CSV

Même nom de fichier et mêmes colonnes que dans les notebooks, pour qu'ils relisent ces runs directement.

In [ ]:
def lancer(cadre, creer, data, L, E, frac, seed, p=0.1, out="resultats"):
    fichier = f"{out}/{cadre}_w{L}_e{E}_frac{frac}_s{seed}.csv"
    if os.path.exists(fichier):
        return
    torch.manual_seed(seed)
    model, opt = creer(L)
    model.to(device)
    hist = entrainer(model, opt, data, E, t_early=round(frac * E), p=p, batch=BATCH)
    hist.assign(cadre=cadre, width=L, epochs=E, frac=frac, seed=seed).to_csv(fichier, index=False)
    print(fichier, " val acc", round(hist.val_acc.tail(5).mean(), 4))

MLP = lambda L: mlp(784, 10, couche_cache=L)
CNN = lambda L: cnn(1, 10, largeur=L)

def MLP_TETE(L):
    model, opt = mlp(784, 10, couche_cache=L)
    model[2] = nn.Identity()      # on enlève le 1er dropout : il reste seulement celui avant la sortie
    return model, opt

## 0. On range les runs faits avec l'ancien `train.py`

L'ancienne version mesurait la loss d'entraînement sur tout le train, la nouvelle sur 10 000 images (et ajoute `train_acc`). Pour les configs qu'on complète, si un seul fichier vient de l'ancienne version, on déplace toute la config dans `resultats/ancien/` et on la relance : sinon on comparerait deux façons de mesurer la loss d'entraînement.

In [ ]:
os.makedirs("resultats/ancien", exist_ok=True)
for motif in ["mlp_fmnist_w16_e50", "cnn_fmnist_w4_e50", "cnn_fmnist_w16_e50"]:
    fichiers = glob.glob(f"resultats/{motif}_frac*_s*.csv")
    vieux = [f for f in fichiers if "train_acc" not in pd.read_csv(f, nrows=1).columns]
    if vieux:
        for f in fichiers:
            os.replace(f, f"resultats/ancien/{os.path.basename(f)}")
    print(f"{motif} : {len(fichiers)} fichiers, {len(vieux)} de l'ancienne version ->", "tout archivé" if vieux else "on garde")

## 1. MLP largeur 16, 50 epochs : grille complète, 5 seeds

In [ ]:
for frac in EARLY_GRID:
    for seed in range(5):
        lancer("mlp_fmnist", MLP, plat, 16, 50, frac, seed)

## 2. CNN largeur 4, 50 epochs : grille complète, 5 seeds (seuls les runs manquants tournent)

In [ ]:
for frac in EARLY_GRID:
    for seed in range(5):
        lancer("cnn_fmnist", CNN, img, 4, 50, frac, seed)

## 3. Pont p = 0.3

Mêmes cadres, seul p change. Les CSV vont dans `resultats/p03/` (les globs des notebooks ne descendent pas dans les sous-dossiers, donc pas de mélange avec p = 0.1). Le run sans dropout ne dépend pas de p : on réutilise `frac 0` des étapes 1 et 2.

In [ ]:
os.makedirs("resultats/p03", exist_ok=True)
for frac in [0.1, 0.3, 1]:
    for seed in range(5):
        lancer("mlp_fmnist", MLP, plat, 16, 50, frac, seed, p=0.3, out="resultats/p03")
        lancer("cnn_fmnist", CNN, img, 4, 50, frac, seed, p=0.3, out="resultats/p03")

## 4. Calibration MLP largeur 32, 50 epochs (il manquait cette case)

In [ ]:
for frac in [0, 1]:
    for seed in [0, 1]:
        lancer("mlp_fmnist", MLP, plat, 32, 50, frac, seed)

## 5. Contrôle CNN largeur 16, 50 epochs (surapprentissage) : frac 0 / 0.1 / 1, 3 seeds

In [ ]:
for frac in [0, 0.1, 1]:
    for seed in [0, 1, 2]:
        lancer("cnn_fmnist", CNN, img, 16, 50, frac, seed)

## 6. (option) MLP avec un seul dropout, avant la sortie, à p = 0.3

C'est la position du dropout dans le notebook CIFAR-10. Sans dropout ce modèle est le même que le MLP normal, donc on réutilise son `frac 0`.

In [ ]:
for frac in [0.1, 0.3, 1]:
    for seed in range(5):
        lancer("mlp_tete_fmnist", MLP_TETE, plat, 16, 50, frac, seed, p=0.3, out="resultats/p03")

## Où on en est

In [ ]:
attendus = {
    "resultats/mlp_fmnist_w16_e50": 5 * len(EARLY_GRID),
    "resultats/cnn_fmnist_w4_e50": 5 * len(EARLY_GRID),
    "resultats/p03/mlp_fmnist_w16_e50": 15,
    "resultats/p03/cnn_fmnist_w4_e50": 15,
    "resultats/mlp_fmnist_w32_e50": 4,
    "resultats/cnn_fmnist_w16_e50": 9,
    "resultats/p03/mlp_tete_fmnist_w16_e50": 15,
}
for motif, n in attendus.items():
    print(f"{motif:42s} {len(glob.glob(motif + '_frac*_s*.csv')):3d} / {n}")

## Analyse du pont p = 0.3

Comme d'habitude : moyenne des 5 dernières epochs, différence seed par seed avec le run sans dropout, IC Student à 95 % (t = 2.776 pour 5 seeds), nombre de seeds où c'est mieux. Pour chaque cadre, p = 0.3 et p = 0.1 sont comparés au même `frac 0`.

In [ ]:
metriques = ["train_loss", "val_loss", "val_acc", "test_acc"]
fin = lambda f: pd.read_csv(f).tail(5)[metriques].mean()

def gains(motif, ref):
    fichiers = [(motif.format(s=s), ref.format(s=s)) for s in range(5)]
    if not all(os.path.exists(a) and os.path.exists(b) for a, b in fichiers):
        return None
    d = pd.DataFrame([fin(a) - fin(b) for a, b in fichiers])
    d[["val_acc", "test_acc"]] *= 100                      # accuracy en points
    ic = 2.776 * d.std() / 5 ** 0.5
    mieux = pd.Series({"train_loss": (d.train_loss < 0).sum(), "val_acc": (d.val_acc > 0).sum()})
    return pd.concat([d.mean(), ic.add_prefix("ic_"), mieux.add_prefix("n_mieux_")])

lignes = {}
for nom, cadre, L in [("MLP", "mlp_fmnist", 16), ("CNN", "cnn_fmnist", 4), ("MLP dropout en tête", "mlp_tete_fmnist", 16)]:
    ref = f"resultats/{cadre.replace('_tete', '')}_w{L}_e50_frac0_s{{s}}.csv"
    for frac in [0.1, 0.3, 1]:
        lignes[(nom, "p = 0.3", frac)] = gains(f"resultats/p03/{cadre}_w{L}_e50_frac{frac}_s{{s}}.csv", ref)
        if "tete" not in cadre:
            lignes[(nom, "p = 0.1", frac)] = gains(f"resultats/{cadre}_w{L}_e50_frac{frac}_s{{s}}.csv", ref)

res = pd.DataFrame({k: v for k, v in lignes.items() if v is not None}).T
res.index.names = ["cadre", "p", "frac"]
res[["train_loss", "ic_train_loss", "n_mieux_train_loss", "val_acc", "ic_val_acc", "n_mieux_val_acc",
     "test_acc", "ic_test_acc"]].round(3)

Comment lire :
- **frac 1** : à p = 0.3 le dropout standard doit perdre encore plus qu'à p = 0.1. Sinon le cadre n'est plus en sous-apprentissage à ce p et le reste ne se lit pas.
- **frac 0.1 et 0.3** : mêmes gains (ou même absence de gain) qu'à p = 0.1 -> p ne change pas nos conclusions, on peut mettre CIFAR-10 à côté de nous. Si early 0.1 se met à perdre à p = 0.3 -> la bonne durée dépend de p, on ne compare les durées entre jeux qu'à p égal.
- **MLP dropout en tête** : si frac 1 est proche de 0, la position compte autant que p, et les résultats CIFAR-10 (un seul dropout) mesurent une intervention plus faible que la nôtre.

## Sauvegarde sur GitHub (token dans les secrets Colab)

In [ ]:
!git config user.name "Felix-prjcts"
!git add -A resultats
!git commit -m "runs complementaires : grille MLP 50 ep relancee, frac 0.05, controles, pont p=0.3"

token = userdata.get("GITHUB_TOKEN")
!git push -q https://{token}@github.com/Felix-prjcts/early-dropout-study.git HEAD 2>&1 | sed "s/{token}/***/g"